In [1]:
# # STEP 02.4 — POSITIVE CROSS-MODAL PAIR CONSTRUCTION

# ## Objective

# Construct scientifically controlled positive cross-modal training pairs
# using the Balanced Class-Conditional Pairing strategy selected in Step 02.3.

# ## Positive Pair Definition

# A positive pair consists of two representations from different modalities
# that belong to the same diagnostic class.

# Examples:

# - Genetic ASD ↔ Behavioral ASD
# - Genetic NON-ASD ↔ Behavioral NON-ASD
# - Genetic ASD ↔ Environmental ASD
# - Behavioral ASD ↔ Environmental ASD

# ## Methodological Principle

# These pairs represent knowledge-level semantic relationships.

# They DO NOT represent:

# - Same patients
# - Shared subject identities
# - Biological sample correspondence
# - Row-index matching across datasets

# ## Pairing Strategy

# Balanced Class-Conditional Sampling:

# - Cross-modal only
# - Same-class only
# - Equal class contribution
# - Equal modality-pair contribution
# - Fixed random seed
# - No exhaustive Cartesian product generation

# ## Expected Output

# A validated positive cross-modal pair dataset containing:

# - Source embedding
# - Target embedding
# - Source modality
# - Target modality
# - Source class
# - Target class
# - Pair label
# - Pair type
# - Sample indices

In [2]:
# =============================================================================
# CELL 2 — IMPORTS AND CONFIGURATION
# =============================================================================

import numpy as np
import pandas as pd
from pathlib import Path

# Reproducibility
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

# =============================================================================
# PROJECT ROOT
# =============================================================================

PROJECT_ROOT = Path(
    r"C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening"
)

# =============================================================================
# OUTPUT DIRECTORIES
# =============================================================================

PAIRING_DIR = (
    PROJECT_ROOT /
    "06_cross_modal_learning" /
    "02_cross_modal_pairing"
)

PAIRS_DIR = PAIRING_DIR / "pairs"
REPORTS_DIR = PAIRING_DIR / "reports"

PAIRS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# PAIRING CONFIGURATION
# =============================================================================

CLASS_NAMES = {
    0: "NON-ASD",
    1: "ASD"
}

PAIR_LABEL = 1
PAIR_TYPE = "Positive"
PAIR_RELATIONSHIP = "Same-Class Cross-Modal"

print("=" * 100)
print("STEP 02.4 — POSITIVE CROSS-MODAL PAIR CONSTRUCTION")
print("=" * 100)

print("\nConfiguration:")
print(f"Random Seed       : {RANDOM_SEED}")
print(f"Pair Label        : {PAIR_LABEL}")
print(f"Pair Type         : {PAIR_TYPE}")
print(f"Relationship      : {PAIR_RELATIONSHIP}")
print(f"Output Directory  : {PAIRS_DIR}")

print("\n✓ INITIALIZATION COMPLETED")

STEP 02.4 — POSITIVE CROSS-MODAL PAIR CONSTRUCTION

Configuration:
Random Seed       : 42
Pair Label        : 1
Pair Type         : Positive
Relationship      : Same-Class Cross-Modal
Output Directory  : C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs

✓ INITIALIZATION COMPLETED


In [4]:
# =====================================================================================
# CELL 3 — LOAD FROZEN 32D EMBEDDINGS
# =====================================================================================

from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 100)
print("STEP 02.4 — LOADING FROZEN MODAL REPRESENTATIONS")
print("=" * 100)

# -------------------------------------------------------------
# Resolve Project Root
# -------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "05_multimodal_fusion").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError(
            "Could not locate project root containing '05_multimodal_fusion'."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

print("\nProject Root:")
print(PROJECT_ROOT)

# -------------------------------------------------------------
# Frozen Baseline Artifact Directory
# -------------------------------------------------------------

FUSION_DIR = PROJECT_ROOT / "05_multimodal_fusion"

# -------------------------------------------------------------
# Define Correct Artifact Paths
# -------------------------------------------------------------

GENETIC_EMBEDDING_PATH = (
    FUSION_DIR / "genetic_32d_normalized.npy"
)

GENETIC_TARGET_PATH = (
    FUSION_DIR / "genetic_targets.npy"
)

BEHAVIORAL_EMBEDDING_PATH = (
    FUSION_DIR / "behavioral_32d_normalized.npy"
)

BEHAVIORAL_TARGET_PATH = (
    FUSION_DIR / "behavioral_targets.npy"
)

ENVIRONMENTAL_EMBEDDING_PATH = (
    FUSION_DIR / "environmental_32d_normalized.npy"
)

ENVIRONMENTAL_TARGET_PATH = (
    FUSION_DIR / "environmental_targets.npy"
)

# -------------------------------------------------------------
# Verify Files Before Loading
# -------------------------------------------------------------

artifact_paths = {
    "Genetic Embeddings": GENETIC_EMBEDDING_PATH,
    "Genetic Targets": GENETIC_TARGET_PATH,
    "Behavioral Embeddings": BEHAVIORAL_EMBEDDING_PATH,
    "Behavioral Targets": BEHAVIORAL_TARGET_PATH,
    "Environmental Embeddings": ENVIRONMENTAL_EMBEDDING_PATH,
    "Environmental Targets": ENVIRONMENTAL_TARGET_PATH
}

print("\nArtifact Verification")
print("-" * 100)

for name, path in artifact_paths.items():
    print(f"{name:<30}: {'FOUND' if path.exists() else 'MISSING'}")

missing_files = [
    str(path)
    for path in artifact_paths.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following required artifacts are missing:\n"
        + "\n".join(missing_files)
    )

# -------------------------------------------------------------
# Load Genetic
# -------------------------------------------------------------

genetic_embeddings = np.load(GENETIC_EMBEDDING_PATH)
genetic_labels = np.load(GENETIC_TARGET_PATH)

# -------------------------------------------------------------
# Load Behavioral
# -------------------------------------------------------------

behavioral_embeddings = np.load(BEHAVIORAL_EMBEDDING_PATH)
behavioral_labels = np.load(BEHAVIORAL_TARGET_PATH)

# -------------------------------------------------------------
# Load Environmental
# -------------------------------------------------------------

environmental_embeddings = np.load(ENVIRONMENTAL_EMBEDDING_PATH)
environmental_labels = np.load(ENVIRONMENTAL_TARGET_PATH)

# -------------------------------------------------------------
# Store Modalities
# -------------------------------------------------------------

modalities = {
    "Genetic": {
        "embeddings": genetic_embeddings,
        "labels": genetic_labels
    },
    "Behavioral": {
        "embeddings": behavioral_embeddings,
        "labels": behavioral_labels
    },
    "Environmental": {
        "embeddings": environmental_embeddings,
        "labels": environmental_labels
    }
}

# -------------------------------------------------------------
# Display Summary
# -------------------------------------------------------------

print("\n" + "=" * 100)
print("FROZEN EMBEDDING SUMMARY")
print("=" * 100)

for modality, data in modalities.items():

    embeddings = data["embeddings"]
    labels = data["labels"]

    print(f"\n{modality.upper()}")
    print("-" * 60)
    print(f"Embedding shape : {embeddings.shape}")
    print(f"Label shape     : {labels.shape}")
    print(f"Dimensions      : {embeddings.shape[1]}")
    print(f"Unique classes  : {np.unique(labels)}")

print("\n" + "=" * 100)
print("✓ FROZEN 32D EMBEDDINGS LOADED SUCCESSFULLY")
print("✓ SAME BASELINE ARTIFACTS USED THROUGHOUT PHASE 01")
print("✓ READY FOR KNOWLEDGE-LEVEL POSITIVE PAIR CONSTRUCTION")
print("=" * 100)

STEP 02.4 — LOADING FROZEN MODAL REPRESENTATIONS

Project Root:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening

Artifact Verification
----------------------------------------------------------------------------------------------------
Genetic Embeddings            : FOUND
Genetic Targets               : FOUND
Behavioral Embeddings         : FOUND
Behavioral Targets            : FOUND
Environmental Embeddings      : FOUND
Environmental Targets         : FOUND

FROZEN EMBEDDING SUMMARY

GENETIC
------------------------------------------------------------
Embedding shape : (116, 32)
Label shape     : (116,)
Dimensions      : 32
Unique classes  : [0 1]

BEHAVIORAL
------------------------------------------------------------
Embedding shape : (880, 32)
Label shape     : (880,)
Dimensions      : 32
Unique classes  : [0 1]

ENVIRONMENTAL
------------------------------------------------------------
Embedding shape : (47768, 32)
Label shape     : (47768,)
Dimensions      

In [5]:
# =============================================================================
# CELL 4 — VALIDATE EMBEDDING AND LABEL INTEGRITY
# =============================================================================

print("=" * 100)
print("EMBEDDING AND LABEL INTEGRITY VALIDATION")
print("=" * 100)

validation_results = []

for modality, data in modalities.items():
    
    embeddings = data["embeddings"]
    labels = data["labels"]
    
    sample_count_match = len(embeddings) == len(labels)
    
    embedding_dimension = (
        embeddings.shape[1]
        if embeddings.ndim == 2
        else None
    )
    
    unique_classes = np.unique(labels)
    
    validation_results.append({
        "Modality": modality,
        "Embedding_Samples": len(embeddings),
        "Label_Samples": len(labels),
        "Sample_Count_Match": sample_count_match,
        "Embedding_Dimension": embedding_dimension,
        "Unique_Classes": list(unique_classes)
    })
    
    print(f"\n{modality}")
    print(f"Embedding samples : {len(embeddings)}")
    print(f"Label samples     : {len(labels)}")
    print(f"Sample match      : {sample_count_match}")
    print(f"Embedding dim     : {embedding_dimension}")
    print(f"Classes           : {unique_classes}")
    
    assert sample_count_match, (
        f"Sample-label mismatch detected in {modality}"
    )
    
    assert embedding_dimension == 32, (
        f"{modality} does not contain 32D embeddings"
    )
    
    assert set(unique_classes).issubset({0, 1}), (
        f"Unexpected class labels detected in {modality}"
    )


integrity_df = pd.DataFrame(validation_results)

print("\n" + "=" * 100)
print("VALIDATION SUMMARY")
print("=" * 100)

display(integrity_df)

print("\n✓ ALL EMBEDDING-LABEL RELATIONSHIPS VALID")
print("✓ ALL MODALITIES USE 32-D REPRESENTATIONS")
print("✓ PAIR CONSTRUCTION CAN PROCEED")

EMBEDDING AND LABEL INTEGRITY VALIDATION

Genetic
Embedding samples : 116
Label samples     : 116
Sample match      : True
Embedding dim     : 32
Classes           : [0 1]

Behavioral
Embedding samples : 880
Label samples     : 880
Sample match      : True
Embedding dim     : 32
Classes           : [0 1]

Environmental
Embedding samples : 47768
Label samples     : 47768
Sample match      : True
Embedding dim     : 32
Classes           : [0 1]

VALIDATION SUMMARY


,Modality,Embedding_Samples,Label_Samples,Sample_Count_Match,Embedding_Dimension,Unique_Classes
0,Genetic,116,116,True,32,"[0, 1]"
1,Behavioral,880,880,True,32,"[0, 1]"
2,Environmental,47768,47768,True,32,"[0, 1]"



✓ ALL EMBEDDING-LABEL RELATIONSHIPS VALID
✓ ALL MODALITIES USE 32-D REPRESENTATIONS
✓ PAIR CONSTRUCTION CAN PROCEED


In [6]:
# =============================================================================
# CELL 5 — CREATE CLASS-SPECIFIC SAMPLE POOLS
# =============================================================================

print("=" * 100)
print("CREATING CLASS-CONDITIONAL SAMPLE POOLS")
print("=" * 100)

class_pools = {}

pool_summary = []

for modality, data in modalities.items():
    
    embeddings = data["embeddings"]
    labels = data["labels"]
    
    class_pools[modality] = {}
    
    for class_label, class_name in CLASS_NAMES.items():
        
        # Find indices belonging to this class
        class_indices = np.where(labels == class_label)[0]
        
        # Store indices
        class_pools[modality][class_label] = class_indices
        
        pool_summary.append({
            "Modality": modality,
            "Class_Label": class_label,
            "Class_Name": class_name,
            "Available_Samples": len(class_indices)
        })
        
        print(
            f"{modality:<15} | "
            f"{class_name:<10} | "
            f"Available Samples: {len(class_indices)}"
        )


pool_summary_df = pd.DataFrame(pool_summary)

print("\n" + "=" * 100)
print("CLASS POOL SUMMARY")
print("=" * 100)

display(pool_summary_df)

CREATING CLASS-CONDITIONAL SAMPLE POOLS
Genetic         | NON-ASD    | Available Samples: 29
Genetic         | ASD        | Available Samples: 87
Behavioral      | NON-ASD    | Available Samples: 439
Behavioral      | ASD        | Available Samples: 441
Environmental   | NON-ASD    | Available Samples: 45650
Environmental   | ASD        | Available Samples: 2118

CLASS POOL SUMMARY


,Modality,Class_Label,Class_Name,Available_Samples
0,Genetic,0,NON-ASD,29
1,Genetic,1,ASD,87
2,Behavioral,0,NON-ASD,439
3,Behavioral,1,ASD,441
4,Environmental,0,NON-ASD,45650
5,Environmental,1,ASD,2118


In [7]:
# =============================================================================
# CELL 6 — DETERMINE BALANCED PAIRING BUDGET
# =============================================================================

print("=" * 100)
print("BALANCED PAIRING BUDGET DETERMINATION")
print("=" * 100)


# -------------------------------------------------------------
# FIND MINIMUM AVAILABLE SAMPLES FOR EACH CLASS
# ACROSS ALL MODALITIES
# -------------------------------------------------------------

class_balance_budget = {}

for class_label, class_name in CLASS_NAMES.items():
    
    available_counts = {
        modality: len(class_pools[modality][class_label])
        for modality in modalities.keys()
    }
    
    minimum_available = min(available_counts.values())
    
    class_balance_budget[class_label] = minimum_available
    
    print(f"\nClass: {class_name}")
    
    for modality, count in available_counts.items():
        print(f"  {modality:<15}: {count}")
    
    print(f"  → Balanced maximum : {minimum_available}")


# -------------------------------------------------------------
# USE A COMMON PAIRING BUDGET ACROSS BOTH CLASSES
# -------------------------------------------------------------

PAIRS_PER_CLASS_PER_MODALITY_COMBINATION = min(
    class_balance_budget.values()
)


print("\n" + "=" * 100)
print("FINAL BALANCED PAIRING BUDGET")
print("=" * 100)

print(
    "Pairs per class per modality combination: "
    f"{PAIRS_PER_CLASS_PER_MODALITY_COMBINATION}"
)

print("\nReason:")
print(
    "A common budget is used to ensure equal contribution "
    "from ASD and NON-ASD classes."
)

print("\n✓ CLASS BALANCE CONSTRAINT ESTABLISHED")

BALANCED PAIRING BUDGET DETERMINATION

Class: NON-ASD
  Genetic        : 29
  Behavioral     : 439
  Environmental  : 45650
  → Balanced maximum : 29

Class: ASD
  Genetic        : 87
  Behavioral     : 441
  Environmental  : 2118
  → Balanced maximum : 87

FINAL BALANCED PAIRING BUDGET
Pairs per class per modality combination: 29

Reason:
A common budget is used to ensure equal contribution from ASD and NON-ASD classes.

✓ CLASS BALANCE CONSTRAINT ESTABLISHED


In [8]:
# =============================================================================
# CELL 7 — DEFINE POSITIVE PAIR CONSTRUCTION FUNCTION
# =============================================================================

def construct_positive_pairs(
    source_modality,
    target_modality,
    class_label,
    n_pairs,
    modalities,
    class_pools,
    rng
):
    """
    Construct balanced same-class cross-modal positive pairs.
    
    Important:
    These pairs represent knowledge-level semantic relationships.
    They do NOT represent patient-level correspondence.
    """
    
    # ---------------------------------------------------------
    # GET AVAILABLE INDICES
    # ---------------------------------------------------------
    
    source_indices = class_pools[source_modality][class_label]
    target_indices = class_pools[target_modality][class_label]
    
    
    # ---------------------------------------------------------
    # VALIDATE AVAILABILITY
    # ---------------------------------------------------------
    
    if len(source_indices) < n_pairs:
        raise ValueError(
            f"Insufficient samples in {source_modality} "
            f"for class {class_label}"
        )
    
    if len(target_indices) < n_pairs:
        raise ValueError(
            f"Insufficient samples in {target_modality} "
            f"for class {class_label}"
        )
    
    
    # ---------------------------------------------------------
    # RANDOM BALANCED SAMPLING
    # WITHOUT REPLACEMENT
    # ---------------------------------------------------------
    
    selected_source_indices = rng.choice(
        source_indices,
        size=n_pairs,
        replace=False
    )
    
    selected_target_indices = rng.choice(
        target_indices,
        size=n_pairs,
        replace=False
    )
    
    
    # ---------------------------------------------------------
    # RETRIEVE EMBEDDINGS
    # ---------------------------------------------------------
    
    source_embeddings = (
        modalities[source_modality]["embeddings"]
        [selected_source_indices]
    )
    
    target_embeddings = (
        modalities[target_modality]["embeddings"]
        [selected_target_indices]
    )
    
    
    # ---------------------------------------------------------
    # CREATE PAIR METADATA
    # ---------------------------------------------------------
    
    pair_metadata = []
    
    for i in range(n_pairs):
        
        pair_metadata.append({
            
            "source_modality": source_modality,
            "target_modality": target_modality,
            
            "source_index": int(selected_source_indices[i]),
            "target_index": int(selected_target_indices[i]),
            
            "source_class": int(class_label),
            "target_class": int(class_label),
            
            "class_name": CLASS_NAMES[class_label],
            
            "pair_label": PAIR_LABEL,
            "pair_type": PAIR_TYPE,
            "relationship": PAIR_RELATIONSHIP
        })
    
    
    return (
        source_embeddings,
        target_embeddings,
        pd.DataFrame(pair_metadata)
    )


print("=" * 100)
print("POSITIVE PAIR CONSTRUCTION FUNCTION DEFINED")
print("=" * 100)

print("""
Function guarantees:

✓ Cross-modal pairing
✓ Same-class relationship
✓ Balanced sample count
✓ Sampling without replacement
✓ Reproducibility through fixed RNG
✓ Complete pair metadata
✓ No patient-level matching
""")

POSITIVE PAIR CONSTRUCTION FUNCTION DEFINED

Function guarantees:

✓ Cross-modal pairing
✓ Same-class relationship
✓ Balanced sample count
✓ Sampling without replacement
✓ Reproducibility through fixed RNG
✓ Complete pair metadata
✓ No patient-level matching



In [9]:
# =============================================================================
# CELL 8 — CONSTRUCT ALL POSITIVE CROSS-MODAL PAIRS
# =============================================================================

print("=" * 100)
print("CONSTRUCTING POSITIVE CROSS-MODAL PAIRS")
print("=" * 100)


# -------------------------------------------------------------
# DEFINE UNIQUE MODALITY COMBINATIONS
# -------------------------------------------------------------

modality_pairs = [
    ("Genetic", "Behavioral"),
    ("Genetic", "Environmental"),
    ("Behavioral", "Environmental")
]


# -------------------------------------------------------------
# STORAGE
# -------------------------------------------------------------

all_source_embeddings = []
all_target_embeddings = []
all_pair_metadata = []


# -------------------------------------------------------------
# CONSTRUCT PAIRS
# -------------------------------------------------------------

for source_modality, target_modality in modality_pairs:
    
    print(
        f"\nConstructing: "
        f"{source_modality} ↔ {target_modality}"
    )
    
    for class_label, class_name in CLASS_NAMES.items():
        
        (
            source_embeddings,
            target_embeddings,
            metadata
        ) = construct_positive_pairs(
            
            source_modality=source_modality,
            target_modality=target_modality,
            class_label=class_label,
            n_pairs=PAIRS_PER_CLASS_PER_MODALITY_COMBINATION,
            modalities=modalities,
            class_pools=class_pools,
            rng=rng
        )
        
        
        all_source_embeddings.append(source_embeddings)
        all_target_embeddings.append(target_embeddings)
        all_pair_metadata.append(metadata)
        
        
        print(
            f"  {class_name:<10} → "
            f"{len(metadata)} positive pairs created"
        )


# -------------------------------------------------------------
# COMBINE ALL PAIRS
# -------------------------------------------------------------

positive_source_embeddings = np.vstack(
    all_source_embeddings
)

positive_target_embeddings = np.vstack(
    all_target_embeddings
)

positive_pair_metadata = pd.concat(
    all_pair_metadata,
    ignore_index=True
)


print("\n" + "=" * 100)
print("POSITIVE PAIR CONSTRUCTION COMPLETED")
print("=" * 100)

print(
    f"\nTotal Positive Pairs: "
    f"{len(positive_pair_metadata)}"
)

print(
    f"Source Embedding Shape: "
    f"{positive_source_embeddings.shape}"
)

print(
    f"Target Embedding Shape: "
    f"{positive_target_embeddings.shape}"
)

print("\nExpected structure:")
print(
    f"{len(positive_pair_metadata)} pairs × "
    f"32D source embeddings × "
    f"32D target embeddings"
)

CONSTRUCTING POSITIVE CROSS-MODAL PAIRS

Constructing: Genetic ↔ Behavioral
  NON-ASD    → 29 positive pairs created
  ASD        → 29 positive pairs created

Constructing: Genetic ↔ Environmental
  NON-ASD    → 29 positive pairs created
  ASD        → 29 positive pairs created

Constructing: Behavioral ↔ Environmental
  NON-ASD    → 29 positive pairs created
  ASD        → 29 positive pairs created

POSITIVE PAIR CONSTRUCTION COMPLETED

Total Positive Pairs: 174
Source Embedding Shape: (174, 32)
Target Embedding Shape: (174, 32)

Expected structure:
174 pairs × 32D source embeddings × 32D target embeddings


In [10]:
# =============================================================================
# CELL 9 — VALIDATE POSITIVE PAIR INTEGRITY AND BALANCE
# =============================================================================

print("=" * 100)
print("POSITIVE PAIR VALIDATION")
print("=" * 100)


# -------------------------------------------------------------
# 1. SAME-CLASS VALIDATION
# -------------------------------------------------------------

same_class_valid = (
    positive_pair_metadata["source_class"]
    ==
    positive_pair_metadata["target_class"]
).all()


# -------------------------------------------------------------
# 2. CROSS-MODAL VALIDATION
# -------------------------------------------------------------

cross_modal_valid = (
    positive_pair_metadata["source_modality"]
    !=
    positive_pair_metadata["target_modality"]
).all()


# -------------------------------------------------------------
# 3. POSITIVE LABEL VALIDATION
# -------------------------------------------------------------

positive_label_valid = (
    positive_pair_metadata["pair_label"] == 1
).all()


# -------------------------------------------------------------
# 4. EMBEDDING DIMENSION VALIDATION
# -------------------------------------------------------------

embedding_dimension_valid = (
    positive_source_embeddings.shape[1] == 32
    and
    positive_target_embeddings.shape[1] == 32
)


# -------------------------------------------------------------
# 5. DUPLICATE EXACT PAIR CHECK
# -------------------------------------------------------------

duplicate_pair_count = (
    positive_pair_metadata[
        [
            "source_modality",
            "target_modality",
            "source_index",
            "target_index",
            "source_class",
            "target_class"
        ]
    ]
    .duplicated()
    .sum()
)


# -------------------------------------------------------------
# 6. CLASS BALANCE
# -------------------------------------------------------------

class_distribution = (
    positive_pair_metadata["class_name"]
    .value_counts()
    .sort_index()
)


# -------------------------------------------------------------
# 7. MODALITY COMBINATION BALANCE
# -------------------------------------------------------------

modality_distribution = (
    positive_pair_metadata
    .groupby(
        ["source_modality", "target_modality"]
    )
    .size()
    .reset_index(name="Pair_Count")
)


# -------------------------------------------------------------
# PRINT RESULTS
# -------------------------------------------------------------

print("\nVALIDATION RESULTS")
print("-" * 100)

print(f"Same-Class Condition Valid     : {same_class_valid}")
print(f"Cross-Modality Condition Valid : {cross_modal_valid}")
print(f"Positive Label Valid           : {positive_label_valid}")
print(f"32D Embedding Valid            : {embedding_dimension_valid}")
print(f"Duplicate Exact Pairs          : {duplicate_pair_count}")


print("\nCLASS DISTRIBUTION")
print("-" * 100)

display(
    class_distribution
    .rename_axis("Class")
    .reset_index(name="Pair_Count")
)


print("\nMODALITY PAIR DISTRIBUTION")
print("-" * 100)

display(modality_distribution)


# -------------------------------------------------------------
# SCIENTIFIC ASSERTIONS
# -------------------------------------------------------------

assert same_class_valid
assert cross_modal_valid
assert positive_label_valid
assert embedding_dimension_valid
assert duplicate_pair_count == 0


print("\n" + "=" * 100)
print("✓ POSITIVE PAIR VALIDATION PASSED")
print("✓ SAME-CLASS RELATIONSHIPS CONFIRMED")
print("✓ CROSS-MODAL CONSTRAINT CONFIRMED")
print("✓ PAIR BALANCE CONFIRMED")
print("✓ NO EXACT DUPLICATE PAIRS DETECTED")
print("=" * 100)

POSITIVE PAIR VALIDATION

VALIDATION RESULTS
----------------------------------------------------------------------------------------------------
Same-Class Condition Valid     : True
Cross-Modality Condition Valid : True
Positive Label Valid           : True
32D Embedding Valid            : True
Duplicate Exact Pairs          : 0

CLASS DISTRIBUTION
----------------------------------------------------------------------------------------------------


,Class,Pair_Count
0,ASD,87
1,NON-ASD,87



MODALITY PAIR DISTRIBUTION
----------------------------------------------------------------------------------------------------


,source_modality,target_modality,Pair_Count
0,Behavioral,Environmental,58
1,Genetic,Behavioral,58
2,Genetic,Environmental,58



✓ POSITIVE PAIR VALIDATION PASSED
✓ SAME-CLASS RELATIONSHIPS CONFIRMED
✓ CROSS-MODAL CONSTRAINT CONFIRMED
✓ PAIR BALANCE CONFIRMED
✓ NO EXACT DUPLICATE PAIRS DETECTED


In [11]:
# =============================================================================
# CELL 10 — SAVE POSITIVE PAIR ARTIFACTS AND FINAL SUMMARY
# =============================================================================

print("=" * 100)
print("SAVING POSITIVE CROSS-MODAL PAIR ARTIFACTS")
print("=" * 100)


# -------------------------------------------------------------
# FILE PATHS
# -------------------------------------------------------------

EMBEDDING_PAIR_PATH = (
    PAIRS_DIR /
    "02_04_positive_cross_modal_embedding_pairs.npz"
)

METADATA_PATH = (
    PAIRS_DIR /
    "02_04_positive_cross_modal_pair_metadata.csv"
)

SUMMARY_PATH = (
    REPORTS_DIR /
    "02_04_positive_pair_construction_summary.csv"
)

CLASS_BALANCE_PATH = (
    REPORTS_DIR /
    "02_04_positive_pair_class_distribution.csv"
)

MODALITY_BALANCE_PATH = (
    REPORTS_DIR /
    "02_04_positive_pair_modality_distribution.csv"
)


# -------------------------------------------------------------
# SAVE EMBEDDING PAIRS
# -------------------------------------------------------------

np.savez_compressed(
    EMBEDDING_PAIR_PATH,
    
    source_embeddings=positive_source_embeddings,
    target_embeddings=positive_target_embeddings,
    
    pair_labels=np.ones(
        len(positive_pair_metadata),
        dtype=np.int64
    )
)


# -------------------------------------------------------------
# SAVE METADATA
# -------------------------------------------------------------

positive_pair_metadata.to_csv(
    METADATA_PATH,
    index=False
)


# -------------------------------------------------------------
# CREATE SUMMARY
# -------------------------------------------------------------

summary_data = pd.DataFrame([
    
    {
        "Metric": "Pairing Strategy",
        "Value": "Balanced Class-Conditional Pairing"
    },
    
    {
        "Metric": "Pair Type",
        "Value": "Positive"
    },
    
    {
        "Metric": "Pair Label",
        "Value": 1
    },
    
    {
        "Metric": "Relationship Definition",
        "Value": "Same-Class Cross-Modal"
    },
    
    {
        "Metric": "Total Positive Pairs",
        "Value": len(positive_pair_metadata)
    },
    
    {
        "Metric": "Pairs Per Class Per Modality Combination",
        "Value": PAIRS_PER_CLASS_PER_MODALITY_COMBINATION
    },
    
    {
        "Metric": "Source Embedding Dimension",
        "Value": positive_source_embeddings.shape[1]
    },
    
    {
        "Metric": "Target Embedding Dimension",
        "Value": positive_target_embeddings.shape[1]
    },
    
    {
        "Metric": "Patient-Level Matching",
        "Value": "Not Used"
    },
    
    {
        "Metric": "Random Seed",
        "Value": RANDOM_SEED
    }
])

summary_data.to_csv(
    SUMMARY_PATH,
    index=False
)


# -------------------------------------------------------------
# SAVE DISTRIBUTIONS
# -------------------------------------------------------------

class_distribution.rename_axis(
    "Class"
).reset_index(
    name="Pair_Count"
).to_csv(
    CLASS_BALANCE_PATH,
    index=False
)

modality_distribution.to_csv(
    MODALITY_BALANCE_PATH,
    index=False
)


# -------------------------------------------------------------
# FINAL OUTPUT
# -------------------------------------------------------------

print(f"\n✓ Embedding pairs saved:\n{EMBEDDING_PAIR_PATH}")

print(f"\n✓ Pair metadata saved:\n{METADATA_PATH}")

print(f"\n✓ Construction summary saved:\n{SUMMARY_PATH}")

print(f"\n✓ Class distribution saved:\n{CLASS_BALANCE_PATH}")

print(f"\n✓ Modality distribution saved:\n{MODALITY_BALANCE_PATH}")


print("\n" + "=" * 100)
print("STEP 02.4 — FINAL POSITIVE PAIR CONSTRUCTION SUMMARY")
print("=" * 100)

print(f"""
Pairing Strategy:
Balanced Class-Conditional Pairing

Pair Type:
Positive

Semantic Relationship:
Same-Class Cross-Modal

Total Positive Pairs:
{len(positive_pair_metadata)}

Pair Construction Budget:
{PAIRS_PER_CLASS_PER_MODALITY_COMBINATION} pairs
per class per modality combination

Modality Relationships:
1. Genetic ↔ Behavioral
2. Genetic ↔ Environmental
3. Behavioral ↔ Environmental

Classes:
1. ASD
2. NON-ASD

Embedding Structure:
Source: {positive_source_embeddings.shape}
Target: {positive_target_embeddings.shape}

Patient-Level Matching:
NOT USED

Scientific Interpretation:
Each pair represents a controlled semantic relationship between
independently learned representations from different modalities.

The pairs indicate same-class knowledge compatibility.

They do NOT indicate measurements from the same patient.
""")

print("=" * 100)
print("✓ STEP 02.4 COMPLETED")
print("✓ POSITIVE CROSS-MODAL PAIRS CONSTRUCTED")
print("✓ CLASS BALANCE PRESERVED")
print("✓ MODALITY BALANCE PRESERVED")
print("✓ KNOWLEDGE-LEVEL METHODOLOGY PRESERVED")
print("✓ NO PATIENT-LEVEL ASSUMPTIONS MADE")
print("=" * 100)

print("\nNEXT LOGICAL STEP:")
print("02.5 — NEGATIVE CROSS-MODAL PAIR CONSTRUCTION")

SAVING POSITIVE CROSS-MODAL PAIR ARTIFACTS

✓ Embedding pairs saved:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\02_04_positive_cross_modal_embedding_pairs.npz

✓ Pair metadata saved:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\02_04_positive_cross_modal_pair_metadata.csv

✓ Construction summary saved:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\reports\02_04_positive_pair_construction_summary.csv

✓ Class distribution saved:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\reports\02_04_positive_pair_class_distribution.csv

✓ Modality distribution saved:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\reports\02_04_positive